# Delivery Time Prediction and Above-Typical Risk Analysis

This notebook trains two complementary models on the Amazon delivery dataset:
- a **regression model** to predict delivery time in minutes
- a **classification model** to predict whether an order is **above the typical delivery duration**

The classification threshold is based on the dataset median delivery time so the target stays balanced and learnable.


## 1. Imports and Configuration


In [1]:
import json
import os
import pickle

import joblib
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import cross_val_score, train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2
REG_TUNING_TRIALS = 25
CLF_TUNING_TRIALS = 25
MAX_DISTANCE_KM = 50
MODEL_DIR = 'best_models'

optuna.logging.set_verbosity(optuna.logging.WARNING)


## 2. Load Data and Inspect the Raw Dataset


In [2]:
df = pd.read_csv('amazon_delivery.csv')
df.columns = df.columns.str.lower().str.strip()

print('Shape:', df.shape)
display(df.head())
display(df.isnull().sum().rename('missing_values').to_frame())


Shape: (43739, 16)


,order_id,agent_age,agent_rating,store_latitude,store_longitude,drop_latitude,drop_longitude,order_date,order_time,pickup_time,weather,traffic,vehicle,area,delivery_time,category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


,missing_values
order_id,0
agent_age,0
agent_rating,54
store_latitude,0
store_longitude,0
drop_latitude,0
drop_longitude,0
order_date,0
order_time,0
pickup_time,0


## 3. Cleaning and Feature Engineering


In [3]:
def haversine(lat1, lon1, lat2, lon2):
    radius_km = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return radius_km * c

df['agent_rating'] = df['agent_rating'].fillna(df['agent_rating'].median())
df['weather'] = df['weather'].fillna(df['weather'].mode()[0])

df['order_date'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d', errors='coerce')
df['order_time'] = pd.to_datetime(df['order_time'], format='%H:%M:%S', errors='coerce')
df['pickup_time'] = pd.to_datetime(df['pickup_time'], format='%H:%M:%S', errors='coerce')

df['distance_km'] = haversine(
    df['store_latitude'],
    df['store_longitude'],
    df['drop_latitude'],
    df['drop_longitude'],
)
df['order_hour'] = df['order_time'].dt.hour
df['order_day'] = df['order_time'].dt.day_name()
df['is_weekend'] = (df['order_time'].dt.weekday >= 5).astype(int)
df['prep_time'] = (df['pickup_time'] - df['order_time']).dt.total_seconds() / 60

display(df[['delivery_time', 'distance_km', 'order_hour', 'prep_time']].describe().T)


,count,mean,std,min,25%,50%,75%,max
delivery_time,43739.0,124.905645,51.915451,10.000000,90.000000,125.00000,160.000000,270.000000
distance_km,43739.0,38.561752,534.564299,1.465067,4.663432,9.22045,13.682379,19692.674606
order_hour,43648.0,17.425976,4.818494,0.000000,15.000000,19.00000,21.000000,23.000000
prep_time,43648.0,-17.325422,196.243803,-1435.000000,5.000000,10.00000,15.000000,15.000000


## 4. Target Definition and Sanity Filtering

Before training, the notebook removes unrealistic route-distance rows so broken coordinates do not distort the model. The classification target is then based on the median delivery time, which keeps the split balanced between shorter and longer deliveries.


In [4]:
invalid_distance_rows = int((df['distance_km'] > MAX_DISTANCE_KM).sum())
df = df[df['distance_km'] <= MAX_DISTANCE_KM].copy()
df = df[df['delivery_time'] < df['delivery_time'].quantile(0.99)].copy()
threshold = float(df['delivery_time'].median())
df['is_delayed'] = (df['delivery_time'] > threshold).astype(int)
df = df.dropna().copy()

print(f'Removed unrealistic routes above {MAX_DISTANCE_KM} km: {invalid_distance_rows}')
print(f'Above-typical threshold: {threshold:.1f} minutes')
display(df[['distance_km', 'delivery_time']].describe().T)
display(df['is_delayed'].value_counts().rename_axis('class').reset_index(name='count'))
display((df['is_delayed'].value_counts(normalize=True) * 100).round(2).rename('percentage').to_frame())


Removed unrealistic routes above 50 km: 188
Above-typical threshold: 125.0 minutes


,count,mean,std,min,25%,50%,75%,max
distance_km,42868.0,9.682934,5.601031,1.465067,4.658009,9.192951,13.631344,20.969489
delivery_time,42868.0,123.051017,49.891818,10.000000,90.000000,125.000000,155.000000,240.000000


,class,count
0,0,23063
1,1,19805


,percentage
is_delayed,
0,53.8
1,46.2


## 5. Prepare Features


In [5]:
encoded_df = pd.get_dummies(
    df,
    columns=['weather', 'traffic', 'vehicle', 'area', 'category', 'order_day'],
    drop_first=True,
)

X = encoded_df.drop(columns=[
    'delivery_time', 'is_delayed',
    'order_id', 'order_date', 'order_time', 'pickup_time',
    'store_latitude', 'store_longitude', 'drop_latitude', 'drop_longitude',
])
y = encoded_df['delivery_time']
y_class = encoded_df['is_delayed']

X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
bool_columns = X.select_dtypes('bool').columns
if len(bool_columns) > 0:
    X = X.astype({col: int for col in bool_columns})

print('Feature count:', X.shape[1])
display(pd.DataFrame({'feature': X.columns, 'dtype': X.dtypes.astype(str)}).head(15))


Feature count: 34


,feature,dtype
agent_age,agent_age,int64
agent_rating,agent_rating,float64
distance_km,distance_km,float64
order_hour,order_hour,float64
is_weekend,is_weekend,int64
prep_time,prep_time,float64
weather_Fog,weather_Fog,int64
weather_Sandstorms,weather_Sandstorms,int64
weather_Stormy,weather_Stormy,int64
weather_Sunny,weather_Sunny,int64


## 6. Train/Test Split


In [6]:
X_train, X_test, y_train, y_test, y_train_class, y_test_class = train_test_split(
    X, y, y_class, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)


Train shape: (34294, 34)
Test shape: (8574, 34)


## 7. Baseline Models


In [7]:
baseline_reg = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1)
baseline_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1)

baseline_reg.fit(X_train, y_train)
baseline_clf.fit(X_train, y_train_class)

baseline_reg_pred = baseline_reg.predict(X_test)
baseline_clf_pred = baseline_clf.predict(X_test)
baseline_clf_proba = baseline_clf.predict_proba(X_test)[:, 1]


In [8]:
baseline_metrics = pd.DataFrame([
    {
        'model': 'Baseline Regression',
        'mae': round(mean_absolute_error(y_test, baseline_reg_pred), 4),
        'rmse': round(np.sqrt(mean_squared_error(y_test, baseline_reg_pred)), 4),
        'r2': round(r2_score(y_test, baseline_reg_pred), 4),
    },
    {
        'model': 'Baseline Classification',
        'accuracy': round(accuracy_score(y_test_class, baseline_clf_pred), 4),
        'f1_weighted': round(f1_score(y_test_class, baseline_clf_pred, average='weighted'), 4),
        'roc_auc': round(roc_auc_score(y_test_class, baseline_clf_proba), 4),
    },
])
display(baseline_metrics.fillna("-"))


,model,mae,rmse,r2,accuracy,f1_weighted,roc_auc
0,Baseline Regression,17.1231,22.0544,0.805,-,-,-
1,Baseline Classification,-,-,-,0.8374,0.8364,0.9206


## 8. Hyperparameter Tuning with Optuna


In [9]:
def regression_objective(trial):
    model = RandomForestRegressor(
        n_estimators=trial.suggest_int('n_estimators', 100, 500),
        max_depth=trial.suggest_int('max_depth', 5, 40),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
        max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    scores = cross_val_score(
        model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error", n_jobs=1
    )
    return -scores.mean()

def classification_objective(trial):
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int('n_estimators', 100, 500),
        max_depth=trial.suggest_int('max_depth', 5, 40),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
        class_weight=trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample', None]),
        max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    scores = cross_val_score(
        model, X_train, y_train_class, cv=5, scoring="f1_weighted", n_jobs=1
    )
    return -scores.mean()


In [10]:
reg_study = optuna.create_study(direction='minimize')
reg_study.optimize(regression_objective, n_trials=REG_TUNING_TRIALS, show_progress_bar=True)

clf_study = optuna.create_study(direction='minimize')
clf_study.optimize(classification_objective, n_trials=CLF_TUNING_TRIALS, show_progress_bar=True)

print('Best regression params:', reg_study.best_params)
print('Best classification params:', clf_study.best_params)


  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Best regression params: {'n_estimators': 229, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None}
Best classification params: {'n_estimators': 490, 'max_depth': 31, 'min_samples_split': 14, 'min_samples_leaf': 8, 'class_weight': None, 'max_features': 'sqrt'}


## 9. Train Tuned Models


In [11]:
best_reg = RandomForestRegressor(**reg_study.best_params, random_state=RANDOM_STATE, n_jobs=1)
best_clf = RandomForestClassifier(**clf_study.best_params, random_state=RANDOM_STATE, n_jobs=1)

best_reg.fit(X_train, y_train)
best_clf.fit(X_train, y_train_class)

best_reg_pred = best_reg.predict(X_test)
best_clf_pred = best_clf.predict(X_test)
best_clf_proba = best_clf.predict_proba(X_test)[:, 1]


## 10. Evaluation


In [12]:
comparison_table = pd.DataFrame([
    {
        'model': 'Baseline Regression',
        'mae': round(mean_absolute_error(y_test, baseline_reg_pred), 4),
        'rmse': round(np.sqrt(mean_squared_error(y_test, baseline_reg_pred)), 4),
        'r2': round(r2_score(y_test, baseline_reg_pred), 4),
    },
    {
        'model': 'Tuned Regression',
        'mae': round(mean_absolute_error(y_test, best_reg_pred), 4),
        'rmse': round(np.sqrt(mean_squared_error(y_test, best_reg_pred)), 4),
        'r2': round(r2_score(y_test, best_reg_pred), 4),
    },
    {
        'model': 'Baseline Classification',
        'accuracy': round(accuracy_score(y_test_class, baseline_clf_pred), 4),
        'f1_weighted': round(f1_score(y_test_class, baseline_clf_pred, average='weighted'), 4),
        'roc_auc': round(roc_auc_score(y_test_class, baseline_clf_proba), 4),
    },
    {
        'model': 'Tuned Classification',
        'accuracy': round(accuracy_score(y_test_class, best_clf_pred), 4),
        'f1_weighted': round(f1_score(y_test_class, best_clf_pred, average='weighted'), 4),
        'roc_auc': round(roc_auc_score(y_test_class, best_clf_proba), 4),
    },
])

display(comparison_table.fillna("-"))


,model,mae,rmse,r2,accuracy,f1_weighted,roc_auc
0,Baseline Regression,17.1231,22.0544,0.805,-,-,-
1,Tuned Regression,16.5483,21.1849,0.8201,-,-,-
2,Baseline Classification,-,-,-,0.8374,0.8364,0.9206
3,Tuned Classification,-,-,-,0.8459,0.8446,0.9261


In [13]:
regression_error_analysis = pd.DataFrame({
    'actual_delivery_time': y_test.reset_index(drop=True),
    'predicted_delivery_time': np.round(best_reg_pred, 2),
})
regression_error_analysis['error'] = regression_error_analysis['actual_delivery_time'] - regression_error_analysis['predicted_delivery_time']
regression_error_analysis['absolute_error'] = regression_error_analysis['error'].abs()

error_bands = pd.cut(
    regression_error_analysis['absolute_error'],
    bins=[-1, 10, 20, 30, 60, np.inf],
    labels=['0-10 min', '10-20 min', '20-30 min', '30-60 min', '60+ min'],
)
display(error_bands.value_counts().sort_index().rename('orders').to_frame())
display(regression_error_analysis.nlargest(10, 'absolute_error'))


,orders
absolute_error,
0-10 min,3255
10-20 min,2437
20-30 min,1622
30-60 min,1182
60+ min,78


,actual_delivery_time,predicted_delivery_time,error,absolute_error
6973,215,114.90,100.10,100.10
1176,240,141.05,98.95,98.95
4886,225,127.44,97.56,97.56
8452,240,148.24,91.76,91.76
3053,220,131.16,88.84,88.84
1042,190,101.82,88.18,88.18
233,210,123.02,86.98,86.98
2522,180,93.21,86.79,86.79
996,170,87.97,82.03,82.03
209,235,153.48,81.52,81.52


In [14]:
classification_report_df = pd.DataFrame(classification_report(y_test_class, best_clf_pred, output_dict=True)).T
classification_report_df = classification_report_df.round(4)
conf_matrix_df = pd.DataFrame(
    confusion_matrix(y_test_class, best_clf_pred),
    index=['Actual typical range', 'Actual above typical'],
    columns=['Predicted typical range', 'Predicted above typical'],
)

display(classification_report_df)
display(conf_matrix_df)


,precision,recall,f1-score,support
0,0.8229,0.9123,0.8653,4651.0000
1,0.8806,0.7673,0.8201,3923.0000
accuracy,0.8459,0.8459,0.8459,0.8459
macro avg,0.8518,0.8398,0.8427,8574.0000
weighted avg,0.8493,0.8459,0.8446,8574.0000


,Predicted typical range,Predicted above typical
Actual typical range,4243,408
Actual above typical,913,3010


In [15]:
reg_feature_importance = pd.Series(best_reg.feature_importances_, index=X_train.columns)
clf_feature_importance = pd.Series(best_clf.feature_importances_, index=X_train.columns)

display(reg_feature_importance.sort_values(ascending=False).head(10).rename('regression_importance').to_frame())
display(clf_feature_importance.sort_values(ascending=False).head(10).rename('classification_importance').to_frame())


,regression_importance
category_Grocery,0.293576
agent_rating,0.189202
traffic_Low,0.107474
distance_km,0.093209
agent_age,0.087316
weather_Sunny,0.062010
weather_Fog,0.039835
weather_Sandstorms,0.025036
weather_Windy,0.020496
vehicle_scooter,0.019729


,classification_importance
agent_rating,0.170771
agent_age,0.148291
traffic_Low,0.113323
category_Grocery,0.107199
distance_km,0.096192
weather_Sunny,0.094893
order_hour,0.070222
traffic_Jam,0.036204
weather_Fog,0.023485
weather_Windy,0.018275


## 11. Save Models and Metadata


In [16]:
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(best_reg, f'{MODEL_DIR}/regressor.pkl')
joblib.dump(best_clf, f'{MODEL_DIR}/classifier.pkl')
pickle.dump(best_reg, open('delivery_model.pkl', 'wb'))
pickle.dump(best_clf, open('delay_model.pkl', 'wb'))

feature_info = {
    'threshold': threshold,
    'regression': {
        'algorithm': 'RandomForestRegressor',
        'best_params': {k: (v if v is not None else 'None') for k, v in reg_study.best_params.items()},
        'metrics': {
            'mae': round(mean_absolute_error(y_test, best_reg_pred), 4),
            'rmse': round(np.sqrt(mean_squared_error(y_test, best_reg_pred)), 4),
            'r2': round(r2_score(y_test, best_reg_pred), 4),
        },
        'baseline_metrics': {
            'mae': round(mean_absolute_error(y_test, baseline_reg_pred), 4),
            'rmse': round(np.sqrt(mean_squared_error(y_test, baseline_reg_pred)), 4),
            'r2': round(r2_score(y_test, baseline_reg_pred), 4),
        },
    },
    'classification': {
        'algorithm': 'RandomForestClassifier',
        'best_params': {k: (v if v is not None else 'None') for k, v in clf_study.best_params.items()},
        'metrics': {
            'accuracy': round(accuracy_score(y_test_class, best_clf_pred), 4),
            'f1_weighted': round(f1_score(y_test_class, best_clf_pred, average='weighted'), 4),
            'roc_auc': round(roc_auc_score(y_test_class, best_clf_proba), 4),
        },
        'baseline_metrics': {
            'accuracy': round(accuracy_score(y_test_class, baseline_clf_pred), 4),
            'f1_weighted': round(f1_score(y_test_class, baseline_clf_pred, average='weighted'), 4),
            'roc_auc': round(roc_auc_score(y_test_class, baseline_clf_proba), 4),
        },
    },
    'features': {
        'names': X_train.columns.tolist(),
        'types': {col: str(dtype) for col, dtype in X_train.dtypes.items()},
    },
}

with open(f'{MODEL_DIR}/feature_info.json', 'w', encoding='utf-8') as file:
    json.dump(feature_info, file, indent=2)

print('Saved tuned artifacts to best_models/ and pickle exports.')


Saved tuned artifacts to best_models/ and pickle exports.


## 12. Final Summary

- The regression model is the main ETA estimator.
- The classification model is a supporting risk signal based on the dataset median threshold.
- Together they provide both a concrete delivery-time estimate and a probability that the trip will take longer than typical.
